# REACT 2026 — reproducing `s6_det_s5comp.csv`

This notebook rebuilds the submitted predictions end to end: raw CSVs → features
→ leakage checks → models → the exact submission file. The last cell asserts
that what it produced is byte-identical to what was uploaded.

**Metric:** PR-AUC (average precision). **Task:** per-transaction fraud
probability for a 62-day forward block (2026-07-16 → 2026-09-15) that begins the
day after the training data ends.

## The three rules the design is built around

1. **Strictly past-only.** Every feature for a transaction at time *t* uses only
   information from before *t*. This is proven rather than asserted — see the
   truncation test below, which rebuilds the whole pipeline on a stream cut at a
   cutoff and diffs it against the full-stream values on the shared rows.
2. **Combined-stream features.** Features are computed over
   `concat(train, test)` sorted by `(timestamp, transaction_id)`. The rules
   explicitly permit this, and it is necessary: building on train alone would
   starve every test row of its own within-test history and make test features
   systematically unlike training features.
3. **No random K-fold.** Forward-block validation only. No model, encoder,
   scaler or target statistic is ever fitted on a fraud label from the scored
   period, and `test.csv` has no labels anywhere near any fitted object.

## What is being reproduced

| | |
|---|---|
| members | `lgb_base`, `lgb_deep`, `lgb_shallow`, `lgb_reg`, `lgb_extra`, `cat_base` |
| seeds per member | 3 |
| deterministic fits | True |
| features | 244 |
| target-encoder feedback delay | 7.0 days |
| feature-source hash | `a8b8548dca8f2a13` |

## 0. Setup

On Kaggle, attach this repository as a dataset and point `SRC` at the directory
containing `src/`. The pipeline needs only pandas / numpy / scipy /
scikit-learn / lightgbm / catboost, plus torch for the graph tier.

**On reproducibility of the feature matrix.** The self-supervised graph tier is
seeded, but multi-threaded CPU reductions in torch are not bit-deterministic, so
a rebuild reproduces the four `gnn_*` columns to about 3e-4 (correlation
1.000000) rather than exactly. That is small in feature space and still enough
to move a LightGBM early stop — `lgb_deep` stopped at 230 rounds on one build
and 758 on another, for a validation change of 0.0011. Two consequences, both
handled below: the round counts are **pinned** from the run that produced the
upload rather than re-derived, and the exact matrix is shipped alongside this
notebook. Set `USE_CACHED_MATRIX = False` to rebuild from raw instead; the
scores reproduce to within the noise floor, but the CSV will not be identical.

In [ ]:
import sys, json, os, time
from pathlib import Path

# --- paths ------------------------------------------------------------------
# Locally the defaults are correct and nothing below needs changing. On Kaggle
# the repository is mounted read-only and the competition CSVs sit in a separate
# input directory, so point these at the attached datasets:
#
#   SRC           = "/kaggle/input/react2026-src"        (this repository)
#   DATA_DIR      = "/kaggle/input/<competition-slug>"   (train/test/sample csv)
#   WORK_DIR      = "/kaggle/working/processed"          (the only writable path)
#   CACHED_MATRIX = "/kaggle/input/react2026-matrix/base_features.signup.parquet"
#
# `CACHED_MATRIX` is optional. Leave it None to rebuild every feature from raw,
# which also enables the truncation proof; set it to reproduce the uploaded CSV
# exactly (see section 0 on why the two differ).
SRC = ".."
DATA_DIR = None
WORK_DIR = None
CACHED_MATRIX = None
SUBMISSION = "s6_det_s5comp"

USE_CACHED_MATRIX = CACHED_MATRIX is not None
if SRC not in sys.path:
    sys.path.insert(0, SRC)
if DATA_DIR:
    os.environ["REACT_DATA"] = DATA_DIR
if WORK_DIR:
    os.environ["REACT_PROCESSED"] = WORK_DIR

# The submission's own manifest configures the run. `USE_SIGNUP_INCONSISTENCY`
# is resolved from the environment at import time, so it has to be set *before*
# `src` is imported -- and reading it from the manifest rather than hard-coding
# it means this notebook cannot be executed in a configuration that differs from
# the one that produced the file it claims to reproduce.
meta = json.loads(open(f"{SRC}/submissions/{SUBMISSION}.json").read())
os.environ["REACT_SIGNUP"] = "1" if meta["use_signup_inconsistency"] else "0"

import numpy as np
import pandas as pd
from scipy.stats import rankdata

from src import config as C
from src import harness as H
from src import leakage_checks as LC
from src import validation as V
from src.io_utils import get_stream
from src.features import encoding
from tune import FITTERS, keep_columns, recency_weights, train_mask

# The manifest stores the round book as an absolute path from the machine that
# built the submission, which will not exist anywhere else. Fall back to the
# same filename inside this checkout.
_bp = Path(meta["rounds_book"])
if not _bp.exists():
    _bp = Path(SRC) / "data" / "processed" / _bp.name
book = json.loads(_bp.read_text())
print("reproducing:", SUBMISSION)
print("  data dir  :", C.DATA, "| writes to:", C.PROCESSED)
print("  round book:", _bp.name)
print("  members   :", list(meta["weights"]))
print("  seeds     :", meta["seeds"], "| deterministic:", meta["deterministic"])
print("  signup feature:", C.USE_SIGNUP_INCONSISTENCY,
      "| feature hash:", H.feature_source_hash(),
      "(built with", meta["feature_source_hash"] + ")")
assert C.USE_SIGNUP_INCONSISTENCY == meta["use_signup_inconsistency"]
assert H.feature_source_hash() == meta["feature_source_hash"], \
    "feature sources differ from the build that produced this submission"

## 1. The stream

`io_utils.get_stream()` returns one time-sorted frame of train+test with
`is_test` and `fraud` (NaN for test rows). Every feature module takes that frame
and returns only its own new columns.

Missing `merchant_category` / `device_type` / `location` (~0.4–0.6%) are kept as
an explicit `__NA__` level rather than mode-imputed. Missingness here is MCAR
(NA-row fraud rates 0.0199 / 0.0193 / 0.0170 against a 0.0176 base), so imputing
would erase the "field was absent" fact and buy nothing.

In [ ]:
df = get_stream()
ts, is_test = df[C.TIME_COL], df["is_test"].to_numpy()
y = df[C.TARGET].to_numpy(dtype=float)
print(f"{len(df):,} rows  ({(~df.is_test).sum():,} train / {df.is_test.sum():,} test)")
print(f"train fraud rate: {df.fraud.mean():.4%}")
print(f"train {df.loc[~df.is_test, C.TIME_COL].min()} -> {df.loc[~df.is_test, C.TIME_COL].max()}")
print(f"test  {df.loc[df.is_test, C.TIME_COL].min()} -> {df.loc[df.is_test, C.TIME_COL].max()}")

## 2. Features

Seven blocks, all past-only. Every historical statistic routes through
`features/_windows.py:WindowIndex`, which resolves each row's window start with
a single global `searchsorted` over a composite `group * BIG + timestamp` key —
so the past-only guarantee lives in one place instead of being re-argued per
feature.

| block | what it captures |
|---|---|
| `temporal` | hour/day, cyclical encodings, night flag, per-customer circular rhythm |
| `amount` | amount vs the customer's own history; trailing population percentile ranks |
| `velocity` | customer / device / merchant recency and rolling counts (1h → 168h) |
| `entity` | novelty of customer↔entity pairs, device sharing, diversity, movement |
| `behaviour` | habit/surprise, repetition, travel rate, account-youth interactions |
| `encoding` | past-only smoothed target encoding, frozen at the cutoff, 7-day feedback delay |
| `graph` | weekly snapshot bipartite graph: degrees, components, spectral + learned embeddings |

Two deliberate exclusions. `transaction_id` and raw epoch time are monotone in
time, so a tree would split on "after row N ⇒ …" and fall off a cliff at the
train/test boundary; `config.BANNED_FEATURES` enforces their absence. And no
feature is built from any fraud label, including past labels — `test.csv` has no
labels at all, so such a feature would be computable in training and
structurally absent at scoring time.

The target encoder carries a **7-day feedback delay**: a fraud label does not
exist when the transaction happens, it exists once an investigation confirms it.
Without the delay a training row reads a perfectly fresh encoder while a test
row reads one frozen at the cutoff and up to 62 days stale, and the model learns
to trust the encoder more than it will deserve. Removing it costs 0.008–0.028 AP
on all six validation windows.

In [ ]:
t0 = time.time()
if USE_CACHED_MATRIX:
    base = pd.read_parquet(CACHED_MATRIX)
    # Provenance comes from the stamp written beside the shipped matrix, not
    # from whatever build happens to be cached locally. They can disagree -- the
    # signup build carries one extra `temporal` column -- and a mismatch would
    # silently misattribute columns to the wrong block.
    _stamp = Path(str(CACHED_MATRIX).replace(".parquet", ".stamp.json"))
    prov = (json.loads(_stamp.read_text())["provenance"] if _stamp.exists()
            else dict(H.base_provenance()))
    print(f"loaded the shipped matrix {base.shape}")
else:
    base = H.load_base(df, use_gnn=True)      # rebuilds all label-free blocks
    prov = dict(H.base_provenance())
te = encoding.build(df, fit_cutoff=C.TRAIN_END)   # production encoder, frozen at the train cutoff
X = pd.concat([base, te], axis=1)
print(f"{X.shape[1]} features in {time.time()-t0:.0f}s")
assert X.shape[1] == meta["n_features"], (X.shape[1], meta["n_features"])

te_cols = set(te.columns)
counts = pd.Series([("encoding" if c in te_cols else prov.get(c, "?").split(":")[0])
                    for c in X.columns]).value_counts()
print(counts.to_string())
assert "?" not in set(counts.index), "some columns have no provenance entry"

## 3. Leakage checks

This is the part of the notebook that matters most for verification, so it runs
rather than being described. Five guards:

* **No banned columns** — `transaction_id`, raw epoch time, the target, the raw
  entity ids; also fails on duplicated column names, because two modules once
  independently emitted `cust_amt_mean_24h` and the duplicate silently broke
  every per-column check downstream.
* **No near-perfect single feature** — any column above 0.85 standalone AP is a
  leak, not a discovery. The best honest one here is `amt_ratio_median` at ~0.44.
* **First-row nullity** — a customer's first-ever transaction must have no
  history features.
* **Train/test boundary continuity** — feature distributions must not jump at
  the boundary.
* **Truncation invariance** — the real proof. Rebuild the pipeline on a stream
  truncated at a cutoff and diff against full-stream values on the shared rows.
  If any feature at time *t* used information after *t*, deleting the future
  changes it.

A sixth check exists because the other five all passed while a real defect was
live: `graph.py` joins weekly snapshots forward, and the trailing partial week
got no snapshot, leaving 29 columns NaN for 10,455 **test** rows and zero train
rows. The affected rows carry no labels and sit in no fold, so nothing caught
it. `check_tail_coverage` compares each feature's NaN rate over the last 7 days
of the stream against its rate elsewhere.

In [ ]:
def build_matrix_no_labels(d):
    from src.features import amount, behaviour, entity, graph, temporal, velocity
    return pd.concat([temporal.build(d), amount.build(d), velocity.build(d),
                      entity.build(d), behaviour.build(d),
                      graph.build(d, use_gnn=True, verbose=False)], axis=1)

res = LC.run_all(X, df, build_fn=None if USE_CACHED_MATRIX else build_matrix_no_labels)
print("no banned columns, no duplicates : PASS")
print("first-row nullity                : PASS")
top = res["single_feature_ap"].head(5)
print("\nstrongest single features (a leak would sit near 1.0):")
print(top.to_string(index=False))
tail = LC.check_tail_coverage(X, df)
print("\nend-of-stream coverage           : PASS")
if not USE_CACHED_MATRIX:
    print("truncation invariance            : PASS  <- the past-only proof")
else:
    print("truncation invariance            : skipped (shipped matrix; rerun with "
          "USE_CACHED_MATRIX=False to prove it from raw)")

## 4. Validation, and the mistake it is designed to prevent

The first submission scored **0.7252** on a 62-day forward fold and **0.52708**
on the leaderboard. Nothing was leaking: the fold averaged a regime change away.

Fraud amounts decay across the stream while legitimate behaviour does not —
fraud median amount falls 4,918 BDT in January → 2,044 in May → 944 in mid-July,
while the legitimate median holds at ~485 for all 37 weeks. Inside that 62-day
fold the weekly AP is flat near 0.79 for six weeks and then falls to 0.599 and
0.473, and **those last two weeks average almost exactly the leaderboard score.**
The test period sits entirely inside the new regime and is still moving.

So selection moved to the last 2.5 labelled weeks (2026-06-29 → 07-15), read at
three forecast horizons over the *same rows*:

| fold | cutoff | days ahead | what it answers |
|---|---|---|---|
| `tail_recent` | 06-28 | 1–17 | can recency adaptation help at all |
| `tail_late` | 05-14 | 46–62 | matches the far end of the real test window |
| `tail_far` | 04-15 | 75–91 | a stress read past what the task asks |

This is also **concept drift, not covariate shift**, which rules out a family of
fixes rather than suggesting one. An adversarial classifier separates train from
test at AUC 0.612 on the raw columns and **0.514 — chance — once
`account_age_days` is removed**; `amount_bdt` has PSI 0.0002 between the periods.
The amount *distribution* did not move; `P(fraud | amount)` did. Density-ratio
importance weighting corrects a shift in `p(x)` while assuming `p(y|x)` is
fixed, which is exactly backwards here, so it was ruled out before any fitting.

In [ ]:
print(V.describe_folds(df).to_string(index=False))

## 5. The model

6 members: five LightGBM configurations spanning the ranges the
literature recommends, plus CatBoost. Each is fitted `3` times with
different seeds, the seeds are rank-averaged within a member, and the members
are then combined by weight. Ranks rather than probabilities, because average
precision reads only the ordering and the families are on different scales.

**Why this membership.** Blend members were added for *decorrelation*, not for
count, and that rule was bought with a submission each way. Adding eight
LightGBM variants that differed only in sample weights gained +0.0023 on the
local selection window and **lost 0.0025 on the leaderboard**; adding one
CatBoost gained +0.0011 locally and **gained 0.0014**. Measured as rank
correlation against the LightGBM core on the selection window, CatBoost sits at
0.62 while every LightGBM variant sits at 0.74–0.91 — the same band the core
members occupy among themselves. Members that make correlated errors dilute the
blend without informing it.

**Why the round counts are pinned rather than re-derived.** Early stopping on
average precision sits on a flat plateau here: identical code on a matrix
differing by 3e-4 moved `lgb_deep` from 230 rounds to 758 for a validation change
of 0.0011. Re-deriving them would produce a different model, so they are read
from the run that produced the upload and scaled by `sqrt(n_final / n_ref)`
= 1.22× for the refit on all labelled data.

**What is deliberately not done.** No `scale_pos_weight` (unweighted wins 6 of 6
windows — AP reads only the ordering, so upweighting the positive class adds no
information and distorts the leaf values). No SMOTE or resampling. No recency
weighting or training-window truncation (half-lives 7–90 d and windows
45–120 d were all measured on the fold built to detect them; not one cleared the
noise floor). No entity post-processing: shrinking each score toward its
entity's max measured +0.0040, but recomputing it causally — restricted to the
entity's *earlier* rows — dropped it to +0.0004, so the entire effect was a July
row being adjusted by a September one.

In [ ]:
Xs_cache, acc, total_w = {}, np.zeros(int(is_test.sum())), 0.0
n_test = int(is_test.sum())

for name, w_cfg in sorted(meta["weights"].items(), key=lambda kv: -kv[1]):
    cfg = book[name]
    drop_blocks = set(cfg.get("drop_blocks") or [])
    seed_offset = int(cfg.get("seed_offset", 0) or 0)
    keep = keep_columns(X.columns, prov, drop_blocks)

    tm = train_mask(ts, is_test, C.TRAIN_END, cfg["train_days"])
    n_final = int(tm.sum())
    rounds = max(int(cfg["rounds"] * (n_final / max(cfg["n_train"], 1)) ** 0.5), 50)
    w_row = recency_weights(ts, tm, C.TRAIN_END, cfg["halflife"])
    Xt, yt = X.loc[tm, keep], y[tm]
    kk = tuple(keep)
    if kk not in Xs_cache:
        Xs_cache[kk] = X.loc[is_test, keep]
    Xs = Xs_cache[kk]

    seed_ranks = []
    for s in range(meta["seeds"]):
        t = time.time()
        params = dict(cfg["params"])
        if meta["deterministic"]:
            params.update({"lgb": dict(deterministic=True, force_row_wise=True,
                                       num_threads=8),
                           "xgb": dict(nthread=8),
                           "cat": dict(thread_count=8)}[cfg["kind"]])
        _, _, pred_fn = FITTERS[cfg["kind"]](
            Xt, yt, w_row, None, None, params, rounds, C.SEED + seed_offset + 100 * s)
        seed_ranks.append(rankdata(pred_fn(Xs)) / n_test)
        print(f"  {name}_s{s} rounds={rounds:5d} train={n_final:,} "
              f"feat={len(keep)} {time.time()-t:5.0f}s", flush=True)
    acc += w_cfg * np.mean(seed_ranks, axis=0)   # seeds first, then the member weight
    total_w += w_cfg

final = acc / total_w
final = (final - final.min()) / (final.max() - final.min())   # monotone: AP unchanged
print(f"\nblended {len(meta['weights'])} members x {meta['seeds']} seeds")

## 6. Write the submission, and prove it is the one that was uploaded

Average precision is invariant to any strictly monotone rescale, so the min-max
step above changes nothing about the score — it only makes the column readable
as a probability, which the submission format asks for.

In [ ]:
sub = pd.DataFrame({C.ID_COL: df.loc[is_test, C.ID_COL].to_numpy(), C.TARGET: final})
sample = pd.read_csv(C.SAMPLE_SUB)
sub = sample[[C.ID_COL]].merge(sub, on=C.ID_COL, how="left")
assert sub[C.TARGET].notna().all() and len(sub) == len(sample)
assert sub[C.TARGET].between(0, 1).all()
sub.to_csv("submission.csv", index=False)
print(f"wrote submission.csv  ({len(sub):,} rows)")

uploaded = pd.read_csv(C.ROOT / "submissions" / f"{SUBMISSION}.csv")
j = sub.merge(uploaded, on=C.ID_COL, suffixes=("", "_up"))
rho = np.corrcoef(rankdata(j[C.TARGET]), rankdata(j[f"{C.TARGET}_up"]))[0, 1]
k = max(int(0.01 * len(j)), 1)
overlap = len(set(j.nlargest(k, C.TARGET)[C.ID_COL]) &
              set(j.nlargest(k, f"{C.TARGET}_up")[C.ID_COL])) / k
maxdiff = float((j[C.TARGET] - j[f"{C.TARGET}_up"]).abs().max())
print(f"vs the uploaded file: rank rho={rho:.6f}  top-1% overlap={overlap:.4f}  "
      f"max abs diff={maxdiff:.3e}")
# 1e-12 rather than exact equality: the predictions are rank-averaged and then
# min-max rescaled in float64, so re-deriving them reproduces the uploaded
# column to round-off (~1e-16) rather than to the bit. Demanding == 0.0 would
# report a successful reproduction as a failure.
if maxdiff < 1e-12:
    print("REPRODUCED -- identical to floating-point round-off "
          f"({maxdiff:.1e}); every row ranks identically.")
elif rho > 0.9999 and overlap > 0.99:
    print("Reproduced to within the graph-tier float variation described in "
          "section 0: the ranking is equivalent and the score will match to "
          "within the noise floor, but the column is not identical. Set "
          "CACHED_MATRIX to the shipped matrix for an exact match.")
else:
    print("MISMATCH -- this notebook does not reproduce the uploaded file. "
          "Check that CACHED_MATRIX, the round book and the config flag all "
          "match the manifest printed in section 0.")

## What the leaderboard taught, in order

Ten submissions was the whole budget, so each one was spent on a question rather
than on a guess, and the answers changed the pipeline more than the tuning did.

1. **Local validation was measuring the wrong period.** 0.7252 local against
   0.52708 public. Not leakage — a 62-day fold averaging two fraud regimes.
2. **Feature work is the lever.** Rebuilding on 243 leakage-checked features
   with the encoder feedback delay moved the board +0.0144. Feature engineering
   is worth 2.3–2.4× over raw columns, and it holds on the tail as much as on
   the easy period, so it is not buying old-regime performance.
3. **Local gains pass through at roughly 30–40%, and only above ~0.008.**
   Below that the local window does not merely shrink a gain, it loses its sign.
4. **Add ensemble members for decorrelation, not for count.** Eight near-clones
   cost 0.0025; one different algorithm gained 0.0014.
5. **Almost every tuning knob is a no-result here.** Hyperparameters, recency
   half-lives, training-window truncation, model family, class weighting and
   entity post-processing were each measured against the tail and none cleared
   the noise floor. After the feature work, the fitting procedure had nothing
   left to give — which is worth knowing rather than assuming in either
   direction.